# Loan Restructuring Data Cleaning & Standardization

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import os

# Load the primary table
file_path = 'Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'
table_3_5 = pd.read_excel(file_path)

# Load another table for reusability demonstration
file_path_2 = '13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx'
table_restructuring = pd.read_excel(file_path_2)

print("Data loaded successfully.")

## Initial Inspection

In [ ]:
def inspect_table(df, name):
    print(f"--- Inspection: {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:10]}...")
    print("First 5 rows:")
    display(df.head(5))

inspect_table(table_3_5, "table_3_5")

## First Column Validation & Fix

In [ ]:
def validate_first_column(df):
    """
    Inspects the first column. Deletes if all NaN. 
    Replaces NaN with mean if numeric.
    """
    if df.empty:
        return df
    
    first_col = df.columns[0]
    
    # Check if entire column is blank / NaN
    if df[first_col].isna().all():
        df = df.drop(columns=[first_col])
        print(f"Dropped first column: {first_col} (all NaN)")
    else:
        # If numeric, replace NaN with mean
        if pd.api.types.is_numeric_dtype(df[first_col]):
            mean_val = df[first_col].mean()
            df[first_col] = df[first_col].fillna(mean_val)
            print(f"Replaced NaNs in first column with mean: {mean_val}")
        else:
            print("First column is not numeric, leaving NaNs as-is.")
            
    return df

table_3_5 = validate_first_column(table_3_5)

## Unnamed Column Renaming

In [ ]:
def infer_column_names(df):
    """
    Detects Unnamed columns and renames them using financial semantics 
    inferred from data values.
    """
    new_columns = list(df.columns)
    
    # Financial semantic mapping
    semantic_map = {
        'occupation': 'occupation',
        'accounts': 'noOfAccounts',
        'limit': 'creditLimit',
        'outstanding': 'amountOutstanding',
        'restructured': 'restructuredAmount',
        'loan': 'loanId',
        'balance': 'outstandingBalance'
    }
    
    for i, col in enumerate(new_columns):
        if "Unnamed" in str(col):
            inferred_name = None
            # Search first 15 rows for a descriptive string
            for val in df.iloc[:15, i]:
                val_str = str(val).lower()
                if pd.notna(val) and val_str.strip() != "" and not val_str.strip().startswith("("):
                    # Check against semantic map
                    matched = False
                    for key, mapped_name in semantic_map.items():
                        if key in val_str:
                            inferred_name = mapped_name
                            matched = True
                            break
                    if matched: break
                    
                    # If no semantic match, use the string itself if it looks like a header
                    if len(val_str) > 2 and not val_str.replace('.','').replace('-','').isdigit():
                        inferred_name = val_str.strip()
                        break
            
            if inferred_name:
                new_columns[i] = inferred_name
            else:
                # Default fallbacks if nothing inferred
                if i == 0: new_columns[i] = "loanId"
                elif i == 1: new_columns[i] = "restructuredAmount"
                else: new_columns[i] = f"financialDataCol{i}"
                
    df.columns = new_columns
    return df

table_3_5 = infer_column_names(table_3_5)

## Row-Level Cleaning

In [ ]:
def clean_row_levels(df):
    """
    Deletes row index 2.
    Renames columns based on row index 1 if they are inappropriately named.
    """
    # Rename based on row 1 before potentially dropping it
    if 1 in df.index:
        row_1_vals = df.loc[1]
        new_cols = list(df.columns)
        for i, val in enumerate(row_1_vals):
            if pd.notna(val) and str(val).strip() != "" and ("financialDataCol" in str(new_cols[i]) or "Unnamed" in str(new_cols[i])):
                new_cols[i] = str(val).strip()
        df.columns = new_cols
        print("Updated column names based on Row 1 values.")

    # Delete row with index 2
    if 2 in df.index:
        df = df.drop(index=2)
        print("Deleted row with index 2.")
        
    return df

table_3_5 = clean_row_levels(table_3_5)

## Column Name Standardization

In [ ]:
def to_camel_case(text):
    """Converts string to camelCase, removing special characters and preserving existing case transitions."""
    if pd.isna(text) or text == "":
        return "unnamedColumn"
    
    text = str(text)
    # Split by non-alphanumeric and also by camelCase transitions
    words = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', text)
    if not words:
        words = re.sub(r'[^a-zA-Z0-9]', ' ', text).split()
        
    if not words:
        return "unnamedColumn"
    
    # Process words: first is lowercase, rest are capitalized
    processed = [words[0].lower()]
    for word in words[1:]:
        processed.append(word.capitalize())
        
    return "".join(processed)

def standardize_columns(df):
    """Standardizes all column names to camelCase."""
    df.columns = [to_camel_case(col) for col in df.columns]
    return df

table_3_5 = standardize_columns(table_3_5)

## Final Cleaned Output

In [ ]:
def process_all_tables(tables_dict):
    """Applies the full cleaning pipeline to multiple tables."""
    cleaned_tables = {}
    for name, df in tables_dict.items():
        print(f"\nProcessing {name}...")
        df = validate_first_column(df)
        df = clean_row_levels(df)
        df = infer_column_names(df)
        df = standardize_columns(df)
        cleaned_tables[name] = df
        print(f"Finished {name}. Final columns: {df.columns.tolist()[:5]}...")
    return cleaned_tables

# Apply to other tables for reusability demonstration
all_tables = {
    "table_3_5": pd.read_excel('Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'),
    "table_restructuring": pd.read_excel('13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx')
}

cleaned_data = process_all_tables(all_tables)

# Quality Checks
def run_quality_checks(df, name):
    print(f"\n--- Quality Checks for {name} ---")
    unnamed_exists = any("unnamed" in col.lower() for col in df.columns)
    row_2_exists = 2 in df.index
    first_col_all_nan = df.iloc[:, 0].isna().all() if not df.empty else False
    
    print(f"No Unnamed columns: {not unnamed_exists}")
    print(f"Row index 2 removed: {not row_2_exists}")
    print(f"First column is not all NaN: {not first_col_all_nan}")
    
    # Strict camelCase check (no spaces, no underscores, starts with lowercase)
    camel_case_pattern = r'^[a-z][a-zA-Z0-9]*$'
    all_camel = all(re.match(camel_case_pattern, col) for col in df.columns if col)
    print(f"All columns camelCase: {all_camel}")
    if not all_camel:
        bad_cols = [col for col in df.columns if not re.match(camel_case_pattern, col)]
        print(f"Non-camelCase columns: {bad_cols}")

run_quality_checks(cleaned_data['table_3_5'], "table_3_5")
run_quality_checks(cleaned_data['table_restructuring'], "table_restructuring")

# Display final result for table_3_5
print("\nFinal Cleaned Table 3.5 Head:")
display(cleaned_data['table_3_5'].head())